In [0]:
%sql
USE CATALOG esg_assurance;


In [0]:
%sql
USE SCHEMA adi_assurance;

In [0]:
%sql
SHOW TABLES;

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_ids,
    COUNT(DISTINCT po_number) AS distinct_po_numbers
FROM bronze_supplier_procurement;

In [0]:
USE CATALOG esg_assurance;
USE SCHEMA adi_assurance;

In [0]:
SELECT current_catalog(), current_schema();

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS distinct_transaction_ids,
    COUNT(DISTINCT po_number) AS distinct_po_numbers
FROM bronze_supplier_procurement;

In [0]:
SELECT
    transaction_id,
    COUNT(*) AS record_count
FROM bronze_supplier_procurement
GROUP BY transaction_id
HAVING COUNT(*) > 1;

In [0]:
SELECT *
FROM bronze_supplier_procurement
WHERE transaction_id IN (
    SELECT transaction_id
    FROM bronze_supplier_procurement
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
)
ORDER BY transaction_id;

In [0]:
SELECT
    transaction_id,
    SUM(calculated_emissions_tco2e) AS total_emissions_tco2e,
    COUNT(*) AS record_count
FROM bronze_supplier_procurement
GROUP BY transaction_id
HAVING COUNT(*) > 1;

In [0]:
CREATE OR REPLACE TEMP VIEW v_duplicate_test AS
SELECT
    *,
    CASE
        WHEN COUNT(*) OVER (PARTITION BY transaction_id) > 1
        THEN 1
        ELSE 0
    END AS dq_duplicate_fail
FROM bronze_supplier_procurement;

In [0]:
SELECT
    transaction_id,
    po_number,
    supplier_id,
    spend_usd,
    calculated_emissions_tco2e,
    dq_duplicate_fail
FROM v_duplicate_test
WHERE dq_duplicate_fail = 1;

In [0]:
SELECT *
FROM bronze_supplier_procurement
WHERE supplier_id IS NULL
   OR TRIM(supplier_id) = '';

In [0]:
SELECT
    COUNT(*) AS missing_supplier_id_count
FROM bronze_supplier_procurement
WHERE supplier_id IS NULL
   OR TRIM(supplier_id) = '';

In [0]:
SELECT
    p.transaction_id,
    p.supplier_id,
    p.supplier_name,
    p.supplier_country,
    s.supplier_id AS matched_supplier_id,
    s.supplier_name AS master_supplier_name,
    s.supplier_country AS master_supplier_country
FROM bronze_supplier_procurement p
LEFT JOIN bronze_supplier_master s
    ON p.supplier_id = s.supplier_id
WHERE s.supplier_id IS NULL;

In [0]:
SELECT
    p.transaction_id,
    p.supplier_id,
    p.supplier_name,
    p.supplier_country AS transaction_country,
    s.supplier_country AS master_country
FROM bronze_supplier_procurement p
JOIN bronze_supplier_master s
    ON p.supplier_id = s.supplier_id
WHERE p.supplier_country <> s.supplier_country;

In [0]:
SELECT
    transaction_id,
    invoice_date,
    fiscal_year
FROM bronze_supplier_procurement
WHERE TO_DATE(invoice_date) < DATE('2024-11-03')
   OR TO_DATE(invoice_date) > DATE('2025-11-01');

In [0]:
SELECT
    transaction_id,
    invoice_date,
    fiscal_year
FROM bronze_supplier_procurement
WHERE fiscal_year = 'FY2025'
  AND (
        TO_DATE(invoice_date) < DATE('2024-11-03')
        OR TO_DATE(invoice_date) > DATE('2025-11-01')
      );

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_validation AS

SELECT
    p.*,

    /* CTRL-DQ-001: Duplicate transaction */
    CASE
        WHEN COUNT(*) OVER (
            PARTITION BY p.transaction_id
        ) > 1
        THEN 1
        ELSE 0
    END AS dq_duplicate_fail,

    /* CTRL-DQ-002A: Missing supplier ID */
    CASE
        WHEN p.supplier_id IS NULL
          OR TRIM(p.supplier_id) = ''
        THEN 1
        ELSE 0
    END AS dq_supplier_missing_fail,

    /* CTRL-DQ-002B: Supplier country mismatch */
    CASE
        WHEN s.supplier_id IS NOT NULL
         AND p.supplier_country <> s.supplier_country
        THEN 1
        ELSE 0
    END AS dq_supplier_country_mismatch_fail,

    /* CTRL-DQ-003: FY2025 cutoff */
    CASE
        WHEN TO_DATE(p.invoice_date) < DATE('2024-11-03')
          OR TO_DATE(p.invoice_date) > DATE('2025-11-01')
        THEN 1
        ELSE 0
    END AS dq_cutoff_fail

FROM bronze_supplier_procurement p

LEFT JOIN bronze_supplier_master s
    ON p.supplier_id = s.supplier_id;

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_validation AS

SELECT
    x.*,

    CASE
        WHEN dq_duplicate_fail = 1
          OR dq_supplier_missing_fail = 1
          OR dq_supplier_country_mismatch_fail = 1
          OR dq_cutoff_fail = 1
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS validation_status

FROM (

    SELECT
        p.*,

        CASE
            WHEN COUNT(*) OVER (
                PARTITION BY p.transaction_id
            ) > 1
            THEN 1 ELSE 0
        END AS dq_duplicate_fail,

        CASE
            WHEN p.supplier_id IS NULL
              OR TRIM(p.supplier_id) = ''
            THEN 1 ELSE 0
        END AS dq_supplier_missing_fail,

        CASE
            WHEN s.supplier_id IS NOT NULL
             AND p.supplier_country <> s.supplier_country
            THEN 1 ELSE 0
        END AS dq_supplier_country_mismatch_fail,

        CASE
            WHEN TO_DATE(p.invoice_date) < DATE('2024-11-03')
              OR TO_DATE(p.invoice_date) > DATE('2025-11-01')
            THEN 1 ELSE 0
        END AS dq_cutoff_fail

    FROM bronze_supplier_procurement p

    LEFT JOIN bronze_supplier_master s
        ON p.supplier_id = s.supplier_id

) x;

In [0]:
SELECT
    transaction_id,
    supplier_id,
    supplier_country,
    invoice_date,
    dq_duplicate_fail,
    dq_supplier_missing_fail,
    dq_supplier_country_mismatch_fail,
    dq_cutoff_fail,
    validation_status
FROM silver_supplier_procurement_validation
WHERE validation_status = 'REVIEW'
ORDER BY transaction_id;

In [0]:
SELECT
    SUM(dq_duplicate_fail) AS duplicate_exceptions,
    SUM(dq_supplier_missing_fail) AS missing_supplier_exceptions,
    SUM(dq_supplier_country_mismatch_fail) AS supplier_country_mismatch_exceptions,
    SUM(dq_cutoff_fail) AS cutoff_exceptions,
    COUNT(*) AS population
FROM silver_supplier_procurement_validation;

In [0]:
SELECT *
FROM bronze_category_mapping
ORDER BY gl_account;

In [0]:
SELECT
    p.transaction_id,
    p.gl_account,
    p.procurement_category AS source_procurement_category,
    p.scope3_category AS source_scope3_category,
    m.procurement_category AS mapped_procurement_category,
    m.scope3_category AS mapped_scope3_category
FROM bronze_supplier_procurement p
LEFT JOIN bronze_category_mapping m
    ON p.gl_account = m.gl_account
WHERE p.scope3_category <> m.scope3_category
   OR p.procurement_category <> m.procurement_category;

In [0]:
SELECT
    p.transaction_id,
    p.gl_account,
    p.procurement_category,
    p.scope3_category
FROM bronze_supplier_procurement p
LEFT JOIN bronze_category_mapping m
    ON p.gl_account = m.gl_account
WHERE m.gl_account IS NULL;

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_validation AS

SELECT
    x.*,

    CASE
        WHEN dq_duplicate_fail = 1
          OR dq_supplier_missing_fail = 1
          OR dq_supplier_country_mismatch_fail = 1
          OR dq_cutoff_fail = 1
          OR dq_classification_fail = 1
          OR dq_gl_unmapped_fail = 1
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS validation_status

FROM (

    SELECT
        p.*,

        /* CTRL-DQ-001 */
        CASE
            WHEN COUNT(*) OVER (
                PARTITION BY p.transaction_id
            ) > 1
            THEN 1 ELSE 0
        END AS dq_duplicate_fail,

        /* CTRL-DQ-002A */
        CASE
            WHEN p.supplier_id IS NULL
              OR TRIM(p.supplier_id) = ''
            THEN 1 ELSE 0
        END AS dq_supplier_missing_fail,

        /* CTRL-DQ-002B */
        CASE
            WHEN s.supplier_id IS NOT NULL
             AND p.supplier_country <> s.supplier_country
            THEN 1 ELSE 0
        END AS dq_supplier_country_mismatch_fail,

        /* CTRL-DQ-003 */
        CASE
            WHEN TO_DATE(p.invoice_date) < DATE('2024-11-03')
              OR TO_DATE(p.invoice_date) > DATE('2025-11-01')
            THEN 1 ELSE 0
        END AS dq_cutoff_fail,

        /* CTRL-DQ-004 */
        CASE
            WHEN m.gl_account IS NOT NULL
             AND (
                    p.scope3_category <> m.scope3_category
                 OR p.procurement_category <> m.procurement_category
                 )
            THEN 1 ELSE 0
        END AS dq_classification_fail,

        /* CTRL-DQ-005 */
        CASE
            WHEN m.gl_account IS NULL
            THEN 1 ELSE 0
        END AS dq_gl_unmapped_fail

    FROM bronze_supplier_procurement p

    LEFT JOIN bronze_supplier_master s
        ON p.supplier_id = s.supplier_id

    LEFT JOIN bronze_category_mapping m
        ON p.gl_account = m.gl_account

) x;

In [0]:
SELECT
    transaction_id,
    gl_account,
    procurement_category,
    scope3_category,
    dq_classification_fail,
    dq_gl_unmapped_fail,
    validation_status
FROM silver_supplier_procurement_validation
WHERE dq_classification_fail = 1
   OR dq_gl_unmapped_fail = 1;

In [0]:
SELECT
    transaction_id,
    spend_original,
    currency,
    fx_rate,
    spend_usd
FROM bronze_supplier_procurement
WHERE currency <> 'USD'
  AND fx_rate IS NULL;

In [0]:
SELECT
    transaction_id,
    spend_original,
    currency,
    fx_rate,
    spend_usd,
    ROUND(spend_original * fx_rate, 2) AS recalculated_spend_usd,
    ROUND(
        spend_usd - (spend_original * fx_rate),
        2
    ) AS fx_difference
FROM bronze_supplier_procurement
WHERE fx_rate IS NOT NULL
  AND ABS(
        spend_usd - (spend_original * fx_rate)
      ) > 0.01;

In [0]:
SELECT
    transaction_id,
    calculation_method,
    spend_usd,
    supplier_emission_factor,
    secondary_emission_factor,
    calculated_emissions_tco2e,

    CASE
        WHEN calculation_method = 'supplier-specific'
        THEN spend_usd * supplier_emission_factor / 1000

        WHEN calculation_method = 'spend-based'
        THEN spend_usd * secondary_emission_factor / 1000

        ELSE NULL
    END AS reperformed_emissions_tco2e

FROM bronze_supplier_procurement;

In [0]:
SELECT
    transaction_id,
    calculation_method,
    spend_usd,
    supplier_emission_factor,
    secondary_emission_factor,
    calculated_emissions_tco2e,

    CASE
        WHEN calculation_method = 'supplier-specific'
        THEN spend_usd * supplier_emission_factor / 1000

        WHEN calculation_method = 'spend-based'
        THEN spend_usd * secondary_emission_factor / 1000

        ELSE NULL
    END AS reperformed_emissions_tco2e,

    ABS(
        calculated_emissions_tco2e -
        CASE
            WHEN calculation_method = 'supplier-specific'
            THEN spend_usd * supplier_emission_factor / 1000

            WHEN calculation_method = 'spend-based'
            THEN spend_usd * secondary_emission_factor / 1000

            ELSE NULL
        END
    ) AS emissions_difference

FROM bronze_supplier_procurement
WHERE ABS(
        calculated_emissions_tco2e -
        CASE
            WHEN calculation_method = 'supplier-specific'
            THEN spend_usd * supplier_emission_factor / 1000

            WHEN calculation_method = 'spend-based'
            THEN spend_usd * secondary_emission_factor / 1000

            ELSE NULL
        END
    ) > 0.001;

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_validation AS

SELECT
    x.*,

    CASE
        WHEN dq_duplicate_fail = 1
          OR dq_supplier_missing_fail = 1
          OR dq_supplier_country_mismatch_fail = 1
          OR dq_cutoff_fail = 1
          OR dq_classification_fail = 1
          OR dq_gl_unmapped_fail = 1
          OR dq_fx_missing_fail = 1
          OR dq_fx_recalc_fail = 1
          OR dq_emissions_recalc_fail = 1
        THEN 'REVIEW'
        ELSE 'PASS'
    END AS validation_status

FROM (

    SELECT
        p.*,

        CASE
            WHEN COUNT(*) OVER (PARTITION BY p.transaction_id) > 1
            THEN 1 ELSE 0
        END AS dq_duplicate_fail,

        CASE
            WHEN p.supplier_id IS NULL
              OR TRIM(p.supplier_id) = ''
            THEN 1 ELSE 0
        END AS dq_supplier_missing_fail,

        CASE
            WHEN s.supplier_id IS NOT NULL
             AND p.supplier_country <> s.supplier_country
            THEN 1 ELSE 0
        END AS dq_supplier_country_mismatch_fail,

        CASE
            WHEN TO_DATE(p.invoice_date) < DATE('2024-11-03')
              OR TO_DATE(p.invoice_date) > DATE('2025-11-01')
            THEN 1 ELSE 0
        END AS dq_cutoff_fail,

        CASE
            WHEN m.gl_account IS NOT NULL
             AND (
                    p.scope3_category <> m.scope3_category
                 OR p.procurement_category <> m.procurement_category
                 )
            THEN 1 ELSE 0
        END AS dq_classification_fail,

        CASE
            WHEN m.gl_account IS NULL
            THEN 1 ELSE 0
        END AS dq_gl_unmapped_fail,

        CASE
            WHEN p.currency <> 'USD'
             AND p.fx_rate IS NULL
            THEN 1 ELSE 0
        END AS dq_fx_missing_fail,

        CASE
            WHEN p.fx_rate IS NOT NULL
             AND ABS(
                    p.spend_usd -
                    (p.spend_original * p.fx_rate)
                 ) > 0.01
            THEN 1 ELSE 0
        END AS dq_fx_recalc_fail,

        CASE
            WHEN ABS(
                p.calculated_emissions_tco2e -
                CASE
                    WHEN p.calculation_method = 'supplier-specific'
                    THEN p.spend_usd * p.supplier_emission_factor / 1000

                    WHEN p.calculation_method = 'spend-based'
                    THEN p.spend_usd * p.secondary_emission_factor / 1000

                    ELSE NULL
                END
            ) > 0.001
            THEN 1 ELSE 0
        END AS dq_emissions_recalc_fail

    FROM bronze_supplier_procurement p

    LEFT JOIN bronze_supplier_master s
        ON p.supplier_id = s.supplier_id

    LEFT JOIN bronze_category_mapping m
        ON p.gl_account = m.gl_account

) x;

In [0]:
SELECT
    transaction_id,
    currency,
    spend_original,
    fx_rate,
    spend_usd,
    calculated_emissions_tco2e,
    dq_fx_missing_fail,
    dq_fx_recalc_fail,
    dq_emissions_recalc_fail,
    validation_status
FROM silver_supplier_procurement_validation
WHERE dq_fx_missing_fail = 1
   OR dq_fx_recalc_fail = 1
   OR dq_emissions_recalc_fail = 1
ORDER BY transaction_id;

In [0]:
SELECT *
FROM bronze_emission_factor_master
ORDER BY factor_id;

In [0]:
SELECT
    p.transaction_id,
    p.factor_id,
    p.factor_source,
    p.factor_version,
    f.factor_id AS matched_factor_id
FROM bronze_supplier_procurement p
LEFT JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id
WHERE f.factor_id IS NULL;

In [0]:
SELECT
    p.transaction_id,
    p.factor_id,
    f.version,
    f.status
FROM bronze_supplier_procurement p
JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id
WHERE f.status <> 'APPROVED';

In [0]:
SELECT
    p.transaction_id,
    p.invoice_date,
    p.factor_id,
    f.effective_date,
    f.expiration_date,
    f.status
FROM bronze_supplier_procurement p
JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id
WHERE TO_DATE(p.invoice_date) < TO_DATE(f.effective_date)
   OR TO_DATE(p.invoice_date) > TO_DATE(f.expiration_date);

In [0]:
SELECT
    p.transaction_id,
    p.supplier_country,
    p.procurement_category,
    p.factor_id,

    f.country AS factor_country,
    f.procurement_category AS factor_procurement_category

FROM bronze_supplier_procurement p
JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id

WHERE p.supplier_country <> f.country
   OR p.procurement_category <> f.procurement_category;

In [0]:
SELECT
    transaction_id,
    spend_usd,
    factor_id,
    factor_unit,
    calculation_method
FROM bronze_supplier_procurement
WHERE calculation_method = 'spend-based'
  AND factor_unit <> 'kgCO2e/USD';

In [0]:
SELECT
    p.transaction_id,
    p.factor_id,
    p.factor_unit AS transaction_factor_unit,
    f.factor_unit AS master_factor_unit
FROM bronze_supplier_procurement p
JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id
WHERE p.factor_unit <> f.factor_unit;

In [0]:
SELECT
    p.transaction_id,
    p.factor_id,
    p.factor_source,
    p.factor_version,
    p.factor_unit AS transaction_factor_unit,

    f.source AS master_factor_source,
    f.version AS master_factor_version,
    f.factor_unit AS master_factor_unit,
    f.status AS master_factor_status

FROM bronze_supplier_procurement p

LEFT JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id;

In [0]:
SELECT
    p.transaction_id,
    p.factor_id,
    p.factor_source,
    p.factor_version,
    p.factor_unit AS transaction_factor_unit,

    f.source AS master_factor_source,
    f.version AS master_factor_version,
    f.factor_unit AS master_factor_unit,
    f.status AS master_factor_status

FROM bronze_supplier_procurement p

LEFT JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id

WHERE f.factor_id IS NULL
   OR f.status <> 'APPROVED'
   OR p.factor_unit <> f.factor_unit
   OR p.factor_version <> f.version;

In [0]:
SELECT
    p.transaction_id,
    p.calculation_method,
    p.primary_data_available,
    p.evidence_id,
    e.evidence_id AS matched_evidence_id,
    e.verification_status,
    e.document_status
FROM bronze_supplier_procurement p
LEFT JOIN bronze_evidence_register e
    ON p.evidence_id = e.evidence_id
WHERE p.calculation_method = 'supplier-specific'
  AND (
       p.evidence_id IS NULL
       OR TRIM(p.evidence_id) = ''
       OR e.evidence_id IS NULL
      );

In [0]:
SELECT
    p.transaction_id,
    p.evidence_id,
    e.verification_status,
    e.document_status
FROM bronze_supplier_procurement p
JOIN bronze_evidence_register e
    ON p.evidence_id = e.evidence_id
WHERE p.calculation_method = 'supplier-specific'
  AND (
       e.document_status <> 'Available'
       OR e.verification_status IS NULL
      );

In [0]:
SELECT
    transaction_id,
    manual_override,
    override_reason,
    reviewer_status,
    calculated_emissions_tco2e
FROM bronze_supplier_procurement
WHERE manual_override = 'Y';

In [0]:
SELECT
    transaction_id,
    manual_override,
    override_reason,
    reviewer_status,
    calculated_emissions_tco2e
FROM bronze_supplier_procurement
WHERE manual_override = 'Y'
  AND (
       reviewer_status IS NULL
       OR reviewer_status <> 'APPROVED'
      );

In [0]:
SELECT
    transaction_id,
    reviewer_status
FROM bronze_supplier_procurement
WHERE reviewer_status IS NULL
   OR TRIM(reviewer_status) = '';

In [0]:
CREATE OR REPLACE TABLE assurance_control_results AS

/* CTRL-DQ-001 — Duplicate transactions */
SELECT
    'CTRL-DQ-001' AS control_id,
    'Duplicate transaction test' AS control_name,
    'Occurrence / Accuracy' AS assertion,
    COUNT(*) AS population,
    SUM(CASE WHEN duplicate_count > 1 THEN 1 ELSE 0 END) AS exceptions,
    CASE
        WHEN SUM(CASE WHEN duplicate_count > 1 THEN 1 ELSE 0 END) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS control_result
FROM (
    SELECT
        transaction_id,
        COUNT(*) OVER (PARTITION BY transaction_id) AS duplicate_count
    FROM bronze_supplier_procurement
) x

UNION ALL

/* CTRL-DQ-002 — Missing supplier ID */
SELECT
    'CTRL-DQ-002',
    'Missing supplier identifier',
    'Completeness / Traceability',
    COUNT(*),
    SUM(
        CASE
            WHEN supplier_id IS NULL
              OR TRIM(supplier_id) = ''
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN supplier_id IS NULL
                  OR TRIM(supplier_id) = ''
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement

UNION ALL

/* CTRL-DQ-003 — Supplier-country master mismatch */
SELECT
    'CTRL-DQ-003',
    'Supplier master country consistency',
    'Accuracy / Master Data',
    COUNT(*),
    SUM(
        CASE
            WHEN s.supplier_id IS NOT NULL
             AND p.supplier_country <> s.supplier_country
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN s.supplier_id IS NOT NULL
                 AND p.supplier_country <> s.supplier_country
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement p
LEFT JOIN bronze_supplier_master s
    ON p.supplier_id = s.supplier_id

UNION ALL

/* CTRL-DQ-004 — FY2025 cutoff */
SELECT
    'CTRL-DQ-004',
    'FY2025 cutoff test',
    'Cutoff',
    COUNT(*),
    SUM(
        CASE
            WHEN TO_DATE(invoice_date) < DATE('2024-11-03')
              OR TO_DATE(invoice_date) > DATE('2025-11-01')
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN TO_DATE(invoice_date) < DATE('2024-11-03')
                  OR TO_DATE(invoice_date) > DATE('2025-11-01')
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement

UNION ALL

/* CTRL-DQ-005 — Scope 3 classification */
SELECT
    'CTRL-DQ-005',
    'Scope 3 category classification',
    'Classification',
    COUNT(*),
    SUM(
        CASE
            WHEN m.gl_account IS NOT NULL
             AND (
                 p.scope3_category <> m.scope3_category
                 OR p.procurement_category <> m.procurement_category
             )
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN m.gl_account IS NOT NULL
                 AND (
                     p.scope3_category <> m.scope3_category
                     OR p.procurement_category <> m.procurement_category
                 )
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement p
LEFT JOIN bronze_category_mapping m
    ON p.gl_account = m.gl_account

UNION ALL

/* CTRL-DQ-006 — Missing FX */
SELECT
    'CTRL-DQ-006',
    'Foreign-currency FX completeness',
    'Accuracy / Valuation',
    COUNT(*),
    SUM(
        CASE
            WHEN currency <> 'USD'
             AND fx_rate IS NULL
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN currency <> 'USD'
                 AND fx_rate IS NULL
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement

UNION ALL

/* CTRL-DQ-007 — FX reperformance */
SELECT
    'CTRL-DQ-007',
    'USD spend recalculation',
    'Accuracy / Reperformance',
    COUNT(*),
    SUM(
        CASE
            WHEN fx_rate IS NOT NULL
             AND ABS(spend_usd - (spend_original * fx_rate)) > 0.01
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN fx_rate IS NOT NULL
                 AND ABS(spend_usd - (spend_original * fx_rate)) > 0.01
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement

UNION ALL

/* CTRL-DQ-008 — Emissions reperformance */
SELECT
    'CTRL-DQ-008',
    'Emissions calculation reperformance',
    'Accuracy / Reperformance',
    COUNT(*),
    SUM(
        CASE
            WHEN ABS(
                calculated_emissions_tco2e -
                CASE
                    WHEN calculation_method = 'supplier-specific'
                    THEN spend_usd * supplier_emission_factor / 1000

                    WHEN calculation_method = 'spend-based'
                    THEN spend_usd * secondary_emission_factor / 1000

                    ELSE NULL
                END
            ) > 0.001
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN ABS(
                    calculated_emissions_tco2e -
                    CASE
                        WHEN calculation_method = 'supplier-specific'
                        THEN spend_usd * supplier_emission_factor / 1000

                        WHEN calculation_method = 'spend-based'
                        THEN spend_usd * secondary_emission_factor / 1000

                        ELSE NULL
                    END
                ) > 0.001
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement

UNION ALL

/* CTRL-DQ-009 — Factor approval status */
SELECT
    'CTRL-DQ-009',
    'Emission factor approval status',
    'Accuracy / Change Control',
    COUNT(*),
    SUM(
        CASE
            WHEN f.status <> 'APPROVED'
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN f.status <> 'APPROVED'
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement p
JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id

UNION ALL

/* CTRL-DQ-010 — Factor unit */
SELECT
    'CTRL-DQ-010',
    'Emission factor unit consistency',
    'Accuracy',
    COUNT(*),
    SUM(
        CASE
            WHEN p.factor_unit <> f.factor_unit
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN p.factor_unit <> f.factor_unit
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement p
JOIN bronze_emission_factor_master f
    ON p.factor_id = f.factor_id

UNION ALL

/* CTRL-DQ-011 — Evidence coverage */
SELECT
    'CTRL-DQ-011',
    'Supplier-specific evidence coverage',
    'Traceability / Evidence',
    COUNT(*),
    SUM(
        CASE
            WHEN p.calculation_method = 'supplier-specific'
             AND (
                 p.evidence_id IS NULL
                 OR TRIM(p.evidence_id) = ''
                 OR e.evidence_id IS NULL
             )
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN p.calculation_method = 'supplier-specific'
                 AND (
                     p.evidence_id IS NULL
                     OR TRIM(p.evidence_id) = ''
                     OR e.evidence_id IS NULL
                 )
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement p
LEFT JOIN bronze_evidence_register e
    ON p.evidence_id = e.evidence_id

UNION ALL

/* CTRL-DQ-012 — Manual override approval */
SELECT
    'CTRL-DQ-012',
    'Manual override authorization',
    'Authorization / Governance',
    COUNT(*),
    SUM(
        CASE
            WHEN manual_override = 'Y'
             AND (
                 reviewer_status IS NULL
                 OR reviewer_status <> 'APPROVED'
             )
            THEN 1 ELSE 0
        END
    ),
    CASE
        WHEN SUM(
            CASE
                WHEN manual_override = 'Y'
                 AND (
                     reviewer_status IS NULL
                     OR reviewer_status <> 'APPROVED'
                 )
                THEN 1 ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END
FROM bronze_supplier_procurement;

In [0]:
SELECT *
FROM assurance_control_results
ORDER BY control_id;

In [0]:
SELECT
    COUNT(*) AS total_controls,

    SUM(
        CASE
            WHEN control_result = 'PASS'
            THEN 1 ELSE 0
        END
    ) AS passed_controls,

    SUM(
        CASE
            WHEN control_result = 'FAIL'
            THEN 1 ELSE 0
        END
    ) AS failed_controls,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN control_result = 'PASS'
                THEN 1 ELSE 0
            END
        )
        / COUNT(*),
        1
    ) AS control_pass_rate_pct

FROM assurance_control_results;

In [0]:
CREATE OR REPLACE TABLE kpmg_control_pillar_map AS

SELECT 'CTRL-DQ-001' AS control_id, 'Data Management' AS kpmg_pillar
UNION ALL
SELECT 'CTRL-DQ-002', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-003', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-004', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-005', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-006', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-007', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-008', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-009', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-010', 'Data Management'
UNION ALL
SELECT 'CTRL-DQ-011', 'Value Chain'
UNION ALL
SELECT 'CTRL-DQ-012', 'Governance';

In [0]:
SELECT
    m.kpmg_pillar,
    COUNT(*) AS controls,
    SUM(
        CASE
            WHEN r.control_result = 'PASS'
            THEN 1 ELSE 0
        END
    ) AS passed,
    SUM(
        CASE
            WHEN r.control_result = 'FAIL'
            THEN 1 ELSE 0
        END
    ) AS failed

FROM assurance_control_results r

JOIN kpmg_control_pillar_map m
    ON r.control_id = m.control_id

GROUP BY m.kpmg_pillar

ORDER BY m.kpmg_pillar;

In [0]:
CREATE OR REPLACE TABLE assurance_exception_log AS

/* Duplicate */
SELECT
    CONCAT('EXC-DUP-', transaction_id) AS exception_id,
    transaction_id,
    'CTRL-DQ-001' AS control_id,
    'Duplicate transaction' AS exception_type,
    'Occurrence / Accuracy' AS assertion,
    ABS(calculated_emissions_tco2e) AS emissions_exposure_tco2e,
    'HIGH' AS initial_severity,
    'ESG Data / Finance' AS suggested_owner,
    'OPEN' AS remediation_status,
    'Unresolved duplicate may overstate emissions.' AS issue_description
FROM silver_supplier_procurement_validation
WHERE dq_duplicate_fail = 1

UNION ALL

/* Missing supplier ID */
SELECT
    CONCAT('EXC-SUP-', transaction_id),
    transaction_id,
    'CTRL-DQ-002',
    'Missing supplier identifier',
    'Completeness / Traceability',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Procurement / ESG Data',
    'OPEN',
    'Supplier identifier is missing, breaking source-to-supplier lineage.'
FROM silver_supplier_procurement_validation
WHERE dq_supplier_missing_fail = 1

UNION ALL

/* Supplier country mismatch */
SELECT
    CONCAT('EXC-CTRY-', transaction_id),
    transaction_id,
    'CTRL-DQ-003',
    'Supplier master country mismatch',
    'Accuracy / Master Data',
    ABS(calculated_emissions_tco2e),
    'MEDIUM',
    'Procurement Master Data',
    'OPEN',
    'Transaction country disagrees with supplier master and may affect factor selection.'
FROM silver_supplier_procurement_validation
WHERE dq_supplier_country_mismatch_fail = 1

UNION ALL

/* Cutoff */
SELECT
    CONCAT('EXC-CUT-', transaction_id),
    transaction_id,
    'CTRL-DQ-004',
    'Reporting-period cutoff exception',
    'Cutoff',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Finance / ESG Reporting',
    'OPEN',
    'Transaction assigned to FY2025 falls outside the defined reporting period.'
FROM silver_supplier_procurement_validation
WHERE dq_cutoff_fail = 1

UNION ALL

/* Classification */
SELECT
    CONCAT('EXC-CLASS-', transaction_id),
    transaction_id,
    'CTRL-DQ-005',
    'Scope 3 classification exception',
    'Classification',
    ABS(calculated_emissions_tco2e),
    'MEDIUM',
    'ESG Methodology',
    'OPEN',
    'Reported Scope 3 category disagrees with the controlled GL mapping.'
FROM silver_supplier_procurement_validation
WHERE dq_classification_fail = 1

UNION ALL

/* Missing FX */
SELECT
    CONCAT('EXC-FXMISS-', transaction_id),
    transaction_id,
    'CTRL-DQ-006',
    'Missing FX rate',
    'Accuracy / Valuation',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Finance Data',
    'OPEN',
    'Foreign-currency transaction lacks an FX rate.'
FROM silver_supplier_procurement_validation
WHERE dq_fx_missing_fail = 1

UNION ALL

/* FX recalculation */
SELECT
    CONCAT('EXC-FXREC-', transaction_id),
    transaction_id,
    'CTRL-DQ-007',
    'FX recalculation mismatch',
    'Accuracy / Reperformance',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Finance Data',
    'OPEN',
    'Reported USD spend does not agree with spend_original × fx_rate.'
FROM silver_supplier_procurement_validation
WHERE dq_fx_recalc_fail = 1

UNION ALL

/* Emissions reperformance */
SELECT
    CONCAT('EXC-EMIS-', transaction_id),
    transaction_id,
    'CTRL-DQ-008',
    'Emissions reperformance mismatch',
    'Accuracy / Reperformance',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'ESG Data',
    'OPEN',
    'Reported emissions cannot be independently reproduced from the stored inputs.'
FROM silver_supplier_procurement_validation
WHERE dq_emissions_recalc_fail = 1

UNION ALL

/* Factor status */
SELECT
    CONCAT('EXC-EFSTAT-', transaction_id),
    transaction_id,
    'CTRL-DQ-009',
    'Unapproved or retired emission factor',
    'Accuracy / Change Control',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'ESG Methodology',
    'OPEN',
    'Transaction uses a factor that is not approved for current reporting.'
FROM silver_supplier_procurement_validation
WHERE dq_factor_status_fail = 1

UNION ALL

/* Factor unit */
SELECT
    CONCAT('EXC-EFUNIT-', transaction_id),
    transaction_id,
    'CTRL-DQ-010',
    'Emission-factor unit mismatch',
    'Accuracy',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'ESG Methodology',
    'OPEN',
    'Transaction factor unit disagrees with the controlled factor master.'
FROM silver_supplier_procurement_validation
WHERE dq_factor_unit_fail = 1

UNION ALL

/* Evidence */
SELECT
    CONCAT('EXC-EVID-', transaction_id),
    transaction_id,
    'CTRL-DQ-011',
    'Missing supplier-specific evidence',
    'Traceability / Evidence',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Supplier ESG / ESG Data',
    'OPEN',
    'Supplier-specific calculation lacks a valid supporting evidence record.'
FROM silver_supplier_procurement_validation
WHERE dq_evidence_fail = 1

UNION ALL

/* Manual override */
SELECT
    CONCAT('EXC-OVR-', transaction_id),
    transaction_id,
    'CTRL-DQ-012',
    'Unapproved manual override',
    'Authorization / Governance',
    ABS(calculated_emissions_tco2e),
    'CRITICAL',
    'ESG Reporting Controller',
    'OPEN',
    'Manual override was applied without approved reviewer status.'
FROM silver_supplier_procurement_validation
WHERE dq_override_fail = 1;

In [0]:
DESCRIBE TABLE silver_supplier_procurement_validation;

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_validation_enhanced AS

SELECT
    s.*,

    /* Factor does not exist in master */
    CASE
        WHEN f.factor_id IS NULL
        THEN 1 ELSE 0
    END AS dq_factor_missing_fail,

    /* Factor exists but is not approved */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND f.status <> 'APPROVED'
        THEN 1 ELSE 0
    END AS dq_factor_status_fail,

    /* Factor used outside effective period */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND (
              TO_DATE(s.invoice_date) < TO_DATE(f.effective_date)
              OR TO_DATE(s.invoice_date) > TO_DATE(f.expiration_date)
         )
        THEN 1 ELSE 0
    END AS dq_factor_date_fail,

    /* Factor geography/category mismatch */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND (
              s.supplier_country <> f.country
              OR s.procurement_category <> f.procurement_category
         )
        THEN 1 ELSE 0
    END AS dq_factor_mapping_fail,

    /* Factor unit mismatch */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND s.factor_unit <> f.factor_unit
        THEN 1 ELSE 0
    END AS dq_factor_unit_fail,

    /* Missing evidence for supplier-specific method */
    CASE
        WHEN s.calculation_method = 'supplier-specific'
         AND (
              s.evidence_id IS NULL
              OR TRIM(s.evidence_id) = ''
              OR e.evidence_id IS NULL
         )
        THEN 1 ELSE 0
    END AS dq_evidence_fail,

    /* Manual override without approval */
    CASE
        WHEN s.manual_override = 'Y'
         AND (
              s.reviewer_status IS NULL
              OR s.reviewer_status <> 'APPROVED'
         )
        THEN 1 ELSE 0
    END AS dq_override_fail,

    /* Missing review status */
    CASE
        WHEN s.reviewer_status IS NULL
          OR TRIM(s.reviewer_status) = ''
        THEN 1 ELSE 0
    END AS dq_review_missing_fail

FROM silver_supplier_procurement_validation s

LEFT JOIN bronze_emission_factor_master f
    ON s.factor_id = f.factor_id

LEFT JOIN bronze_evidence_register e
    ON s.evidence_id = e.evidence_id;

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_validation_enhanced AS

SELECT
    s.*,

    /* Factor does not exist in master */
    CASE
        WHEN f.factor_id IS NULL
        THEN 1 ELSE 0
    END AS dq_factor_missing_fail,

    /* Factor exists but is not approved */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND f.status <> 'APPROVED'
        THEN 1 ELSE 0
    END AS dq_factor_status_fail,

    /* Factor used outside effective period */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND (
              TO_DATE(s.invoice_date) < TO_DATE(f.effective_date)
              OR TO_DATE(s.invoice_date) > TO_DATE(f.expiration_date)
         )
        THEN 1 ELSE 0
    END AS dq_factor_date_fail,

    /* Factor geography/category mismatch */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND (
              s.supplier_country <> f.country
              OR s.procurement_category <> f.procurement_category
         )
        THEN 1 ELSE 0
    END AS dq_factor_mapping_fail,

    /* Factor unit mismatch */
    CASE
        WHEN f.factor_id IS NOT NULL
         AND s.factor_unit <> f.factor_unit
        THEN 1 ELSE 0
    END AS dq_factor_unit_fail,

    /* Missing evidence for supplier-specific method */
    CASE
        WHEN s.calculation_method = 'supplier-specific'
         AND (
              s.evidence_id IS NULL
              OR TRIM(s.evidence_id) = ''
              OR e.evidence_id IS NULL
         )
        THEN 1 ELSE 0
    END AS dq_evidence_fail,

    /* Manual override without approval */
    CASE
        WHEN s.manual_override = 'Y'
         AND (
              s.reviewer_status IS NULL
              OR s.reviewer_status <> 'APPROVED'
         )
        THEN 1 ELSE 0
    END AS dq_override_fail,

    /* Missing review status */
    CASE
        WHEN s.reviewer_status IS NULL
          OR TRIM(s.reviewer_status) = ''
        THEN 1 ELSE 0
    END AS dq_review_missing_fail

FROM silver_supplier_procurement_validation s

LEFT JOIN bronze_emission_factor_master f
    ON s.factor_id = f.factor_id

LEFT JOIN bronze_evidence_register e
    ON s.evidence_id = e.evidence_id;

In [0]:
SELECT
    transaction_id,
    dq_factor_missing_fail,
    dq_factor_status_fail,
    dq_factor_date_fail,
    dq_factor_mapping_fail,
    dq_factor_unit_fail,
    dq_evidence_fail,
    dq_override_fail,
    dq_review_missing_fail
FROM silver_supplier_procurement_validation_enhanced
WHERE dq_factor_missing_fail = 1
   OR dq_factor_status_fail = 1
   OR dq_factor_date_fail = 1
   OR dq_factor_mapping_fail = 1
   OR dq_factor_unit_fail = 1
   OR dq_evidence_fail = 1
   OR dq_override_fail = 1
   OR dq_review_missing_fail = 1
ORDER BY transaction_id;

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_hashed AS

SELECT
    *,

    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(transaction_id, ''),
            COALESCE(po_number, ''),
            COALESCE(supplier_id, ''),
            COALESCE(CAST(spend_original AS STRING), ''),
            COALESCE(CAST(spend_usd AS STRING), ''),
            COALESCE(CAST(calculated_emissions_tco2e AS STRING), ''),
            COALESCE(factor_id, ''),
            COALESCE(invoice_date, '')
        ),
        256
    ) AS record_hash

FROM silver_supplier_procurement_validation_enhanced;

In [0]:
SELECT
    transaction_id,
    record_hash,
    po_number,
    supplier_id,
    spend_original,
    spend_usd,
    factor_id,
    calculated_emissions_tco2e
FROM silver_supplier_procurement_hashed
WHERE transaction_id = 'TXN0005';

In [0]:
CREATE OR REPLACE TABLE exception_remediation_actions (
    exception_id STRING,
    transaction_id STRING,
    record_hash STRING,
    action STRING,
    rationale STRING,
    evidence_reference STRING,
    approved_by STRING,
    approval_status STRING,
    remediation_date DATE
);

In [0]:
INSERT INTO exception_remediation_actions
VALUES (
    'EXC-DUP-TXN0005',
    'TXN0005',
    NULL,
    'PENDING_INVESTIGATION',
    'Duplicate transaction identified; correct source record cannot be determined from transaction data alone.',
    'SIMULATED_SOURCE_INVOICE_PENDING',
    NULL,
    'PENDING',
    NULL
);

In [0]:
SELECT *
FROM exception_remediation_actions;

In [0]:
SELECT
    transaction_id,
    record_hash,
    po_number,
    supplier_id,
    spend_original,
    spend_usd,
    calculated_emissions_tco2e,
    factor_id
FROM silver_supplier_procurement_hashed
WHERE transaction_id = 'TXN0005';

In [0]:
UPDATE exception_remediation_actions

SET
    record_hash = '<SECOND_RECORD_FULL_HASH>',
    action = 'EXCLUDE_DUPLICATE',
    rationale = 'Simulated source invoice and ERP review confirm the first TXN0005 record is the valid posting and the second posting is an erroneous duplicate.',
    evidence_reference = 'SIM-EVID-TXN0005-001',
    approved_by = 'ESG_REPORTING_CONTROLLER_SIM',
    approval_status = 'APPROVED',
    remediation_date = CURRENT_DATE()

WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
SELECT *
FROM exception_remediation_actions
WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_remediated AS

SELECT h.*

FROM silver_supplier_procurement_hashed h

LEFT JOIN exception_remediation_actions r
    ON h.record_hash = r.record_hash
   AND r.action = 'EXCLUDE_DUPLICATE'
   AND r.approval_status = 'APPROVED'

WHERE r.record_hash IS NULL;

In [0]:
SELECT
    (SELECT COUNT(*)
     FROM bronze_supplier_procurement) AS bronze_rows,

    (SELECT COUNT(*)
     FROM silver_supplier_procurement_hashed) AS silver_before_rows,

    (SELECT COUNT(*)
     FROM silver_supplier_procurement_remediated) AS silver_after_rows;

In [0]:
SELECT
    transaction_id,
    COUNT(*) AS record_count
FROM silver_supplier_procurement_remediated
GROUP BY transaction_id
HAVING COUNT(*) > 1;

In [0]:
SELECT
    exception_id,
    transaction_id,
    record_hash,
    LENGTH(record_hash) AS hash_length,
    action,
    approval_status
FROM exception_remediation_actions
WHERE transaction_id = 'TXN0005';

In [0]:
UPDATE exception_remediation_actions

SET record_hash = (
    SELECT MAX(record_hash)
    FROM silver_supplier_procurement_hashed
    WHERE transaction_id = 'TXN0005'
      AND ABS(spend_usd - 33246.15) < 0.01
)

WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
SELECT
    exception_id,
    transaction_id,
    record_hash,
    LENGTH(record_hash) AS hash_length,
    action,
    approval_status
FROM exception_remediation_actions
WHERE transaction_id = 'TXN0005';

In [0]:
CREATE OR REPLACE TABLE silver_supplier_procurement_remediated AS

SELECT h.*

FROM silver_supplier_procurement_hashed h

LEFT JOIN exception_remediation_actions r
    ON h.record_hash = r.record_hash
   AND r.action = 'EXCLUDE_DUPLICATE'
   AND r.approval_status = 'APPROVED'

WHERE r.record_hash IS NULL;

In [0]:
SELECT
    transaction_id,
    COUNT(*) AS record_count
FROM silver_supplier_procurement_remediated
GROUP BY transaction_id
HAVING COUNT(*) > 1;

In [0]:
SELECT
    (SELECT COUNT(*) FROM bronze_supplier_procurement) AS bronze_rows,
    (SELECT COUNT(*) FROM silver_supplier_procurement_hashed) AS silver_before_rows,
    (SELECT COUNT(*) FROM silver_supplier_procurement_remediated) AS silver_after_rows;

In [0]:
USE CATALOG esg_assurance;
USE SCHEMA adi_assurance;

In [0]:
SELECT current_catalog(), current_schema();

In [0]:
SHOW TABLES IN esg_assurance.adi_assurance
LIKE 'exception_remediation_actions';

In [0]:
SELECT
    r.exception_id,
    r.transaction_id,
    r.record_hash,
    r.action,
    r.approval_status,
    h.spend_usd,
    h.calculated_emissions_tco2e
FROM esg_assurance.adi_assurance.exception_remediation_actions r
JOIN esg_assurance.adi_assurance.silver_supplier_procurement_hashed h
    ON r.record_hash = h.record_hash
WHERE r.exception_id = 'EXC-DUP-TXN0005';

In [0]:
SELECT
    r.exception_id,
    r.transaction_id,
    r.record_hash,
    r.action,
    r.approval_status,
    h.spend_usd,
    h.calculated_emissions_tco2e
FROM esg_assurance.adi_assurance.exception_remediation_actions r
JOIN esg_assurance.adi_assurance.silver_supplier_procurement_hashed h
    ON r.record_hash = h.record_hash
WHERE r.exception_id = 'EXC-DUP-TXN0005';

In [0]:
SHOW TABLES IN esg_assurance.default;

In [0]:
SHOW TABLES IN workspace.default;

In [0]:
SELECT
    table_catalog,
    table_schema,
    table_name
FROM system.information_schema.tables
WHERE LOWER(table_name) = 'exception_remediation_actions';

In [0]:
SELECT
    r.exception_id,
    r.transaction_id,
    r.record_hash,
    r.action,
    r.approval_status,
    h.spend_usd,
    h.calculated_emissions_tco2e
FROM esg_assurance.adi_assurance.exception_remediation_actions r
JOIN esg_assurance.adi_assurance.silver_supplier_procurement_hashed h
    ON r.record_hash = h.record_hash
WHERE r.exception_id = 'EXC-DUP-TXN0005';

In [0]:
SELECT
    b.transaction_id,
    b.bronze_emissions_tco2e,
    s.remediated_emissions_tco2e,
    b.bronze_emissions_tco2e
      - s.remediated_emissions_tco2e
      AS actual_misstatement_tco2e
FROM (
    SELECT
        transaction_id,
        SUM(calculated_emissions_tco2e) AS bronze_emissions_tco2e
    FROM esg_assurance.adi_assurance.bronze_supplier_procurement
    WHERE transaction_id = 'TXN0005'
    GROUP BY transaction_id
) b
JOIN (
    SELECT
        transaction_id,
        SUM(calculated_emissions_tco2e) AS remediated_emissions_tco2e
    FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
    WHERE transaction_id = 'TXN0005'
    GROUP BY transaction_id
) s
ON b.transaction_id = s.transaction_id;

In [0]:
SELECT
    transaction_id,
    COUNT(*) AS record_count
FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
GROUP BY transaction_id
HAVING COUNT(*) > 1;

In [0]:
SELECT
    b.transaction_id,
    b.bronze_emissions_tco2e,
    s.remediated_emissions_tco2e,
    b.bronze_emissions_tco2e
      - s.remediated_emissions_tco2e
      AS actual_misstatement_tco2e
FROM (
    SELECT
        transaction_id,
        SUM(calculated_emissions_tco2e) AS bronze_emissions_tco2e
    FROM esg_assurance.adi_assurance.bronze_supplier_procurement
    WHERE transaction_id = 'TXN0005'
    GROUP BY transaction_id
) b
JOIN (
    SELECT
        transaction_id,
        SUM(calculated_emissions_tco2e) AS remediated_emissions_tco2e
    FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
    WHERE transaction_id = 'TXN0005'
    GROUP BY transaction_id
) s
ON b.transaction_id = s.transaction_id;

In [0]:
UPDATE esg_assurance.adi_assurance.assurance_exception_log

SET
    remediation_status = 'CLOSED',

    root_cause =
        'Duplicate transaction posting in the source procurement population.',

    remediation_action =
        'Approved duplicate physical record excluded from the remediated Silver layer while the original Bronze source record was preserved.',

    remediation_owner =
        'ESG Data / Finance',

    remediation_due_date =
        CURRENT_DATE(),

    final_disposition =
        'REMEDIATED_AND_RETESTED',

    corrected_emissions_tco2e = (
        SELECT
            SUM(calculated_emissions_tco2e)
        FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
        WHERE transaction_id = 'TXN0005'
    ),

    actual_misstatement_tco2e = (
        SELECT
            before_value - after_value
        FROM
        (
            SELECT
                SUM(calculated_emissions_tco2e) AS before_value
            FROM esg_assurance.adi_assurance.bronze_supplier_procurement
            WHERE transaction_id = 'TXN0005'
        ) b
        CROSS JOIN
        (
            SELECT
                SUM(calculated_emissions_tco2e) AS after_value
            FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
            WHERE transaction_id = 'TXN0005'
        ) s
    )

WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
DESCRIBE TABLE
esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.assurance_exception_log AS

/* Duplicate */
SELECT
    CONCAT('EXC-DUP-', transaction_id) AS exception_id,
    transaction_id,
    'CTRL-DQ-001' AS control_id,
    'Duplicate transaction' AS exception_type,
    'Occurrence / Accuracy' AS assertion,
    ABS(calculated_emissions_tco2e) AS emissions_exposure_tco2e,
    'HIGH' AS initial_severity,
    'ESG Data / Finance' AS suggested_owner,
    'OPEN' AS remediation_status,
    'Unresolved duplicate may overstate emissions.' AS issue_description
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_duplicate_fail = 1

UNION ALL

/* Missing supplier */
SELECT
    CONCAT('EXC-SUP-', transaction_id),
    transaction_id,
    'CTRL-DQ-002',
    'Missing supplier identifier',
    'Completeness / Traceability',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Procurement / ESG Data',
    'OPEN',
    'Supplier identifier is missing and breaks traceability.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_supplier_missing_fail = 1

UNION ALL

/* Country mismatch */
SELECT
    CONCAT('EXC-CTRY-', transaction_id),
    transaction_id,
    'CTRL-DQ-003',
    'Supplier country mismatch',
    'Accuracy / Master Data',
    ABS(calculated_emissions_tco2e),
    'MEDIUM',
    'Procurement Master Data',
    'OPEN',
    'Supplier country disagrees with supplier master.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_supplier_country_mismatch_fail = 1

UNION ALL

/* Cutoff */
SELECT
    CONCAT('EXC-CUT-', transaction_id),
    transaction_id,
    'CTRL-DQ-004',
    'Reporting-period cutoff exception',
    'Cutoff',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Finance / ESG Reporting',
    'OPEN',
    'Transaction falls outside the defined FY2025 reporting period.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_cutoff_fail = 1

UNION ALL

/* Classification */
SELECT
    CONCAT('EXC-CLASS-', transaction_id),
    transaction_id,
    'CTRL-DQ-005',
    'Scope 3 classification exception',
    'Classification',
    ABS(calculated_emissions_tco2e),
    'MEDIUM',
    'ESG Methodology',
    'OPEN',
    'Reported Scope 3 category disagrees with controlled mapping.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_classification_fail = 1

UNION ALL

/* Missing FX */
SELECT
    CONCAT('EXC-FXMISS-', transaction_id),
    transaction_id,
    'CTRL-DQ-006',
    'Missing FX rate',
    'Accuracy / Valuation',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Finance Data',
    'OPEN',
    'Foreign-currency transaction lacks an FX rate.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_fx_missing_fail = 1

UNION ALL

/* FX recalculation */
SELECT
    CONCAT('EXC-FXREC-', transaction_id),
    transaction_id,
    'CTRL-DQ-007',
    'FX recalculation mismatch',
    'Accuracy / Reperformance',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Finance Data',
    'OPEN',
    'USD spend cannot be reproduced from original spend and FX rate.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_fx_recalc_fail = 1

UNION ALL

/* Emissions reperformance */
SELECT
    CONCAT('EXC-EMIS-', transaction_id),
    transaction_id,
    'CTRL-DQ-008',
    'Emissions reperformance mismatch',
    'Accuracy / Reperformance',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'ESG Data',
    'OPEN',
    'Reported emissions cannot be reproduced from stored inputs.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_emissions_recalc_fail = 1

UNION ALL

/* Factor status */
SELECT
    CONCAT('EXC-EFSTAT-', transaction_id),
    transaction_id,
    'CTRL-DQ-009',
    'Unapproved or retired emission factor',
    'Accuracy / Change Control',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'ESG Methodology',
    'OPEN',
    'Transaction uses an emission factor that is not approved.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_factor_status_fail = 1

UNION ALL

/* Factor unit */
SELECT
    CONCAT('EXC-EFUNIT-', transaction_id),
    transaction_id,
    'CTRL-DQ-010',
    'Emission-factor unit mismatch',
    'Accuracy',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'ESG Methodology',
    'OPEN',
    'Transaction factor unit disagrees with controlled factor master.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_factor_unit_fail = 1

UNION ALL

/* Evidence */
SELECT
    CONCAT('EXC-EVID-', transaction_id),
    transaction_id,
    'CTRL-DQ-011',
    'Missing supplier-specific evidence',
    'Traceability / Evidence',
    ABS(calculated_emissions_tco2e),
    'HIGH',
    'Supplier ESG / ESG Data',
    'OPEN',
    'Supplier-specific calculation lacks supporting evidence.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_evidence_fail = 1

UNION ALL

/* Override */
SELECT
    CONCAT('EXC-OVR-', transaction_id),
    transaction_id,
    'CTRL-DQ-012',
    'Unapproved manual override',
    'Authorization / Governance',
    ABS(calculated_emissions_tco2e),
    'CRITICAL',
    'ESG Reporting Controller',
    'OPEN',
    'Manual override was applied without approved reviewer status.'
FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced
WHERE dq_override_fail = 1;

In [0]:
SELECT
    control_id,
    exception_type,
    COUNT(*) AS exception_count
FROM esg_assurance.adi_assurance.assurance_exception_log
GROUP BY control_id, exception_type
ORDER BY control_id;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_exception_log
WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
DELETE FROM esg_assurance.adi_assurance.assurance_exception_log
WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
INSERT INTO esg_assurance.adi_assurance.assurance_exception_log

SELECT
    CONCAT('EXC-DUP-', transaction_id) AS exception_id,
    transaction_id,
    'CTRL-DQ-001' AS control_id,
    'Duplicate transaction' AS exception_type,
    'Occurrence / Accuracy' AS assertion,
    SUM(ABS(calculated_emissions_tco2e)) AS emissions_exposure_tco2e,
    'HIGH' AS initial_severity,
    'ESG Data / Finance' AS suggested_owner,
    'OPEN' AS remediation_status,
    'Multiple physical records share the same transaction ID; the potentially affected emissions are aggregated until investigation determines which record is valid.' AS issue_description

FROM esg_assurance.adi_assurance.silver_supplier_procurement_validation_enhanced

WHERE transaction_id = 'TXN0005'
  AND dq_duplicate_fail = 1

GROUP BY transaction_id;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.assurance_exception_log
WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
ALTER TABLE esg_assurance.adi_assurance.assurance_exception_log
ADD COLUMNS (
    root_cause STRING,
    remediation_action STRING,
    remediation_owner STRING,
    remediation_due_date DATE,
    final_disposition STRING,
    corrected_emissions_tco2e DOUBLE,
    actual_misstatement_tco2e DOUBLE,
    qualitative_materiality STRING
);

In [0]:
UPDATE esg_assurance.adi_assurance.assurance_exception_log

SET
    remediation_status = 'CLOSED',

    root_cause =
        'Duplicate procurement transaction created in the simulated source population.',

    remediation_action =
        'Excluded the approved duplicate physical record from the remediated Silver layer while preserving both original Bronze records.',

    remediation_owner =
        'ESG Data / Finance',

    remediation_due_date =
        CURRENT_DATE(),

    final_disposition =
        'REMEDIATED_AND_RETESTED',

    corrected_emissions_tco2e = (
        SELECT
            SUM(calculated_emissions_tco2e)
        FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
        WHERE transaction_id = 'TXN0005'
    ),

    actual_misstatement_tco2e = (
        SELECT
            SUM(b.calculated_emissions_tco2e)
            -
            SUM(r.calculated_emissions_tco2e)
        FROM esg_assurance.adi_assurance.bronze_supplier_procurement b
        CROSS JOIN esg_assurance.adi_assurance.silver_supplier_procurement_remediated r
        WHERE b.transaction_id = 'TXN0005'
          AND r.transaction_id = 'TXN0005'
    ),

    qualitative_materiality =
        'NO - isolated duplicate in synthetic population; control failure remains relevant for process design.'

WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
SELECT
    exception_id,
    transaction_id,
    emissions_exposure_tco2e,
    corrected_emissions_tco2e,
    actual_misstatement_tco2e,
    remediation_status,
    final_disposition,
    qualitative_materiality
FROM esg_assurance.adi_assurance.assurance_exception_log
WHERE exception_id = 'EXC-DUP-TXN0005';

In [0]:
SELECT
    transaction_id,
    COUNT(*) AS record_count
FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
GROUP BY transaction_id
HAVING COUNT(*) > 1;

In [0]:
SELECT
    (
        SELECT SUM(calculated_emissions_tco2e)
        FROM esg_assurance.adi_assurance.bronze_supplier_procurement
        WHERE transaction_id = 'TXN0005'
    )
    -
    (
        SELECT SUM(calculated_emissions_tco2e)
        FROM esg_assurance.adi_assurance.silver_supplier_procurement_remediated
        WHERE transaction_id = 'TXN0005'
    )
    AS actual_misstatement_tco2e;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.silver_reporting_eligibility AS

SELECT
    s.*,

    CASE
        WHEN e.open_exception_count IS NULL
          OR e.open_exception_count = 0
        THEN 'ELIGIBLE'
        ELSE 'HOLD'
    END AS reporting_eligibility,

    COALESCE(e.open_exception_count, 0)
        AS open_exception_count

FROM
esg_assurance.adi_assurance.silver_supplier_procurement_remediated s

LEFT JOIN (

    SELECT
        transaction_id,
        COUNT(*) AS open_exception_count

    FROM esg_assurance.adi_assurance.assurance_exception_log

    WHERE remediation_status <> 'CLOSED'

    GROUP BY transaction_id

) e

ON s.transaction_id = e.transaction_id;

In [0]:
SELECT
    reporting_eligibility,
    COUNT(*) AS record_count,
    ROUND(
        SUM(calculated_emissions_tco2e),
        3
    ) AS emissions_tco2e

FROM esg_assurance.adi_assurance.silver_reporting_eligibility

GROUP BY reporting_eligibility;

In [0]:
SELECT
    transaction_id,
    supplier_id,
    scope3_category,
    calculated_emissions_tco2e,
    open_exception_count,
    reporting_eligibility

FROM esg_assurance.adi_assurance.silver_reporting_eligibility

WHERE reporting_eligibility = 'HOLD'

ORDER BY transaction_id;

In [0]:
SELECT
    r.transaction_id,
    r.scope3_category,
    r.calculated_emissions_tco2e,
    e.control_id,
    e.exception_type,
    e.initial_severity,
    e.remediation_status

FROM esg_assurance.adi_assurance.silver_reporting_eligibility r

JOIN esg_assurance.adi_assurance.assurance_exception_log e
    ON r.transaction_id = e.transaction_id

WHERE r.reporting_eligibility = 'HOLD'

ORDER BY
    r.transaction_id,
    e.control_id;

In [0]:
SELECT
    scope3_category,
    COUNT(*) AS records,
    ROUND(SUM(calculated_emissions_tco2e), 3) AS emissions_tco2e
FROM esg_assurance.adi_assurance.silver_reporting_eligibility
GROUP BY scope3_category
ORDER BY scope3_category;

In [0]:
SELECT
    reporting_eligibility,
    COUNT(*) AS records,
    ROUND(SUM(calculated_emissions_tco2e), 3) AS emissions_tco2e
FROM esg_assurance.adi_assurance.silver_reporting_eligibility
GROUP BY reporting_eligibility;